# W&B Sweep w/ CNN + MNIST

You will learn:

- Defining a hyperparameter search space: `sweep_config`
- Initializing a sweep: `wandb.sweep`
- Running a sweep agent: `wandb.agent`


In [ ]:
"""
Import libraries.
"""

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import wandb

print("Libraries imported.")

In [ ]:
"""
CNN model class.
"""
class CNN(nn.Module):
    def __init__(self, conv1_filters, conv2_filters, fc_hidden):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, conv1_filters, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),                          # 28x28 -> 14x14
            nn.Conv2d(conv1_filters, conv2_filters, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),                          # 14x14 -> 7x7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(conv2_filters * 7 * 7, fc_hidden),
            nn.ReLU(),
            nn.Linear(fc_hidden, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

print("Model class defined.")

In [ ]:
"""
Load data.
"""

# Download MNIST dataset
full_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transforms.ToTensor())
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transforms.ToTensor())

# Take subset of 20,000 samples
subset, _ = random_split(full_dataset, [20000, len(full_dataset) - 20000])

# 80/20 train-val split
train_set, val_set = random_split(subset, [16000, 4000])

print(f"Train: {len(train_set):,} | Val: {len(val_set):,}")

In [ ]:
"""
Training function.
"""
def train():
    EPOCHS = 5   # shorter training

    # Each trial has its own wandb run
    # Agent populates config (will see in next cells)
    run = wandb.init(
        entity="dataatuci-wandb-demo",
        project="cnn-mnist",
    )
    cfg = wandb.config

    # Define best_val_loss metric
    wandb.define_metric("val/loss", summary="min")

    # Build DataLoaders
    train_loader = DataLoader(train_set, batch_size=cfg.batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_set, batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True)

    # Instantiate model
    model = CNN(conv1_filters=cfg.conv1_filters, conv2_filters=cfg.conv2_filters, fc_hidden=cfg.fc_hidden)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=cfg.learning_rate)

    for epoch in range(EPOCHS):
        # Train
        model.train()
        train_loss, train_correct = 0.0, 0
        for images, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * len(labels)
            train_correct += (outputs.argmax(1) == labels).sum().item()
        train_loss = train_loss / len(train_loader.dataset)
        train_acc = train_correct / len(train_loader.dataset)

        # Validate
        model.eval()
        val_loss, val_correct = 0.0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                outputs = model(images)
                val_loss += criterion(outputs, labels).item() * len(labels)
                val_correct += (outputs.argmax(1) == labels).sum().item()
        val_loss = val_loss / len(val_loader.dataset)
        val_acc = val_correct / len(val_loader.dataset)

        # Log metrics
        wandb.log({
            "epoch": epoch + 1,
            "train/loss": train_loss,
            "train/acc": train_acc,
            "val/loss": val_loss,
            "val/acc": val_acc,
        })

        print(f"Epoch {epoch+1}/{EPOCHS}  train_loss={train_loss:.4f}  train_acc={train_acc:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

    # Finish run
    wandb.finish()

print("Training function defined.")

In [ ]:
"""
Define sweep configuration.
"""

sweep_config = {
    "name": "YOUR_SWEEP_NAME_HERE",                                 # TODO: change this to your sweep name
    "method": "bayes",
    "metric": {"name": "val/loss", "goal": "minimize"},
    "parameters": {
        "learning_rate": {"distribution": "log_uniform_values", "min": 1e-4, "max": 1e-2},
        "batch_size":    {"values": [32, 64, 128]},
        "conv1_filters": {"values": [16, 32, 64]},
        "conv2_filters": {"values": [32, 64, 128]},
        "fc_hidden":     {"values": [64, 128, 256]},
    },
}

print("Sweep config defined.")

In [ ]:
"""
Initialize sweep.
"""

sweep_id = wandb.sweep(
    sweep_config,
    entity="dataatuci-wandb-demo",
    project="cnn-mnist",
)

print(f"Sweep initialized: {sweep_id}")

In [ ]:
"""
Run sweep agent.
"""

wandb.agent(sweep_id, function=train, count=10)